# Validate kael on a CUDA GPU

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JetBerri/kael.cpp/blob/main/scripts/cuda/validate_on_colab.ipynb)

This notebook builds kael with the CUDA backend and runs `scripts/cuda/validate.sh` on the GPU of the runtime. It produces a report archive with one log per step.

| Stage | What runs |
|-------|-----------|
| Build | Release build with `KAEL_ENABLE_CUDA=ON`, examples and benchmarks, plus a second build with `KAEL_USE_CUBLAS=ON` |
| C++ | The full `ctest` suite, the device tests with and without cuBLAS, `compute-sanitizer` and the device benchmarks |
| Models | MNIST on CPU and GPU (at least 97% test accuracy), the character-level GPT and GPT-2 generation |
| Python | The package built for CUDA, the device tests, the full pytest suite and the Python examples on the GPU |

**Requirements.** A GPU runtime: select *Runtime > Change runtime type* and pick any GPU, such as a T4. The run takes 40 to 60 minutes on a T4, most of it compiling with `nvcc`. Run the cells in order, or use *Runtime > Run all*.

## Configuration

In [ ]:
from pathlib import Path

REPOSITORY = "https://github.com/JetBerri/kael.cpp"
REVISION = "main"
CUDA_ARCHITECTURE = "native"
WORKSPACE = Path("/content/kael.cpp")
BUILD_DIRECTORY = "build-cuda"
REPORT_DIRECTORY = "cuda-report"

## Environment

Fails early when the runtime has no GPU, and records the toolchain that the report refers to.

In [ ]:
import shutil
import subprocess

import pandas as pd


def first_line(command):
    if shutil.which(command[0]) is None:
        return "not installed"
    result = subprocess.run(command, capture_output=True, text=True, check=False)
    output = (result.stdout or result.stderr).strip().splitlines()
    return output[0] if output else "no output"


def gpu_description():
    if shutil.which("nvidia-smi") is None:
        return None
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,driver_version,memory.total",
            "--format=csv,noheader",
        ],
        capture_output=True,
        text=True,
        check=False,
    )
    description = result.stdout.strip()
    return description if result.returncode == 0 and description else None


gpu = gpu_description()
if gpu is None:
    raise RuntimeError(
        "No GPU is visible. Select Runtime > Change runtime type, pick a GPU and run again."
    )

nvcc_release = (
    subprocess.run(["nvcc", "--version"], capture_output=True, text=True, check=False)
    .stdout.strip()
    .splitlines()[-1]
)
environment = pd.DataFrame(
    {
        "GPU, driver, memory": [gpu],
        "CUDA compiler": [nvcc_release],
        "CMake": [first_line(["cmake", "--version"])],
        "C++ compiler": [first_line(["c++", "--version"])],
    }
).T.rename(columns={0: "value"})
environment

## Source

In [ ]:
def git(*arguments, cwd=WORKSPACE):
    subprocess.run(["git", *arguments], cwd=cwd, check=True)


if (WORKSPACE / ".git").is_dir():
    git("fetch", "--depth", "1", "origin", REVISION)
    git("checkout", "--force", "FETCH_HEAD")
else:
    git(
        "clone",
        "--depth",
        "1",
        "--branch",
        REVISION,
        REPOSITORY,
        str(WORKSPACE),
        cwd="/",
    )

commit = subprocess.run(
    ["git", "log", "-1", "--format=%h %s"],
    cwd=WORKSPACE,
    capture_output=True,
    text=True,
    check=True,
).stdout.strip()
print(f"validating {commit}")

## Datasets

MNIST, Tiny Shakespeare and the GPT-2 small checkpoint. Each script verifies SHA-256 checksums and skips files that are already present.

In [ ]:
for dataset in ("mnist", "shakespeare", "gpt2"):
    subprocess.run(
        ["bash", f"scripts/download/{dataset}.sh"], cwd=WORKSPACE, check=True
    )

## Validation

Streams the output of `scripts/cuda/validate.sh` as it runs. The exit status is 0 when every step passed, 1 when a step failed and 3 when no step failed but some did not run.

In [ ]:
import os
import sys


def stream(command, environment):
    process = subprocess.Popen(
        command,
        cwd=WORKSPACE,
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    return process.wait()


subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "virtualenv"], check=True
)
build_environment = {**os.environ, "CMAKE_BUILD_PARALLEL_LEVEL": str(os.cpu_count())}
exit_status = stream(
    [
        "bash",
        "scripts/cuda/validate.sh",
        CUDA_ARCHITECTURE,
        BUILD_DIRECTORY,
        REPORT_DIRECTORY,
    ],
    build_environment,
)
print(f"\nvalidate.sh exited with status {exit_status}")

## Results

In [ ]:
import re

REPORT = WORKSPACE / REPORT_DIRECTORY


def read_log(step):
    path = REPORT / f"{step}.log"
    return path.read_text(errors="replace") if path.exists() else ""


def last_match(pattern, text, group=1):
    matches = re.findall(pattern, text)
    if not matches:
        return "not reported"
    match = matches[-1]
    return match[group - 1] if isinstance(match, tuple) else match


def summary_table():
    rows = []
    for line in (REPORT / "summary.txt").read_text().splitlines():
        fields = line.split()
        seconds = (
            fields[2].rstrip("s") if len(fields) > 2 and fields[0] != "SKIP" else ""
        )
        rows.append({"status": fields[0], "step": fields[1], "seconds": seconds})
    return pd.DataFrame(rows)


def key_metrics():
    return pd.DataFrame(
        {
            "C++ suite on the GPU": last_match(
                r"(\d+% tests passed.*)", read_log("ctest")
            ),
            "Device tests": last_match(
                r"\[  PASSED  \] (\d+ tests)", read_log("device-tests")
            ),
            "Device tests with cuBLAS": last_match(
                r"\[  PASSED  \] (\d+ tests)", read_log("device-tests-cublas")
            ),
            "compute-sanitizer": last_match(
                r"ERROR SUMMARY: (\d+ errors?)", read_log("compute-sanitizer")
            ),
            "MNIST accuracy, CPU": last_match(
                r"test accuracy ([\d.]+%)", read_log("mnist-cpu")
            ),
            "MNIST accuracy, GPU": last_match(
                r"test accuracy ([\d.]+%)", read_log("mnist-cuda")
            ),
            "MNIST epoch time, GPU": last_match(
                r"in ([\d.]+ s)", read_log("mnist-cuda")
            ),
            "Device matmul 2048": last_match(
                r"BM_DeviceMatmul/2048\S*.*flops=(\S+)", read_log("benchmarks")
            ),
            "GPT-2 tokens per second": last_match(
                r"([\d.]+) tokens/s", read_log("gpt2-cpp-cuda")
            ),
            "pytest": last_match(r"(\d+ passed[^\n]*)", read_log("python-tests")),
        },
        index=["value"],
    ).T


steps = summary_table()
display(steps)
display(key_metrics())

failed = steps.loc[steps["status"] != "PASS", "step"].tolist()
if exit_status != 0:
    raise RuntimeError(f"validation did not pass, see the logs of: {', '.join(failed)}")
print(f"all {len(steps)} steps passed")

## Report

In [ ]:
gpu_name = gpu.split(",")[0].strip().replace(" ", "-").lower()
revision = commit.split()[0]
archive = shutil.make_archive(
    f"/content/kael-cuda-report-{revision}-{gpu_name}",
    "gztar",
    root_dir=WORKSPACE,
    base_dir=REPORT_DIRECTORY,
)
print(f"report written to {archive}")

try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(archive)